In [1]:
import pandas as pd
import numpy as np

In [2]:
raw_df = pd.read_csv("D:/Study/NYC_Vehicle_Collision_Project/data/nyc_crash_data.csv")

C:\Users\Pradhuman\AppData\Local\Temp\ipykernel_26764\2990149787.py:1: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv("D:/Study/NYC_Vehicle_Collision_Project/data/nyc_crash_data.csv")


In [3]:
raw_df.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,09/11/2021,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,11/01/2023,1:29,BROOKLYN,11230.0,40.62179,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,06/29/2022,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,09/21/2022,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


In [4]:
clean_df = raw_df.copy()

In [5]:
print(f"Raw shape   : {raw_df.shape}")
print(f"Clean shape : {clean_df.shape}")

Raw shape   : (2269187, 29)
Clean shape : (2269187, 29)


In [6]:
baseline = {
    "rows": len(clean_df),
    "columns": clean_df.shape[1],
    "unique_collision_ids": clean_df["COLLISION_ID"].nunique(),
    "duplicate_collision_ids": clean_df["COLLISION_ID"].duplicated().sum()
}

baseline

{'rows': 2269187,
 'columns': 29,
 'unique_collision_ids': 2269187,
 'duplicate_collision_ids': np.int64(0)}

In [8]:
print(clean_df.isna().sum())
print(clean_df.isna().sum().sum())

CRASH DATE                             0
CRASH TIME                             0
BOROUGH                           691375
ZIP CODE                          691661
LATITUDE                          240806
LONGITUDE                         240806
LOCATION                          240806
ON STREET NAME                    499179
CROSS STREET NAME                 870235
OFF STREET NAME                  1862382
NUMBER OF PERSONS INJURED             18
NUMBER OF PERSONS KILLED            8913
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       8202
CONTRIBUTING FACTOR VEHICLE 2     368580
CONTRIBUTING FACTOR VEHICLE 3    2104798
CONTRIBUTING FACTOR VEHICLE 4    2231554
CONTRIBUTING FACTOR VEHICLE 5    2258873
COLLISION_ID                           0
VEHICLE TYPE COD

- There are no duplicate records

**1. Empty Strings to NaN**

In [9]:
clean_df = clean_df.replace(r'^\s*$', pd.NA, regex=True)
print(clean_df.isna().sum())
print(clean_df.isna().sum().sum())

CRASH DATE                             0
CRASH TIME                             0
BOROUGH                           691375
ZIP CODE                          691703
LATITUDE                          240806
LONGITUDE                         240806
LOCATION                          240806
ON STREET NAME                    499194
CROSS STREET NAME                 870252
OFF STREET NAME                  1862411
NUMBER OF PERSONS INJURED             18
NUMBER OF PERSONS KILLED            8913
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       8202
CONTRIBUTING FACTOR VEHICLE 2     368580
CONTRIBUTING FACTOR VEHICLE 3    2104798
CONTRIBUTING FACTOR VEHICLE 4    2231554
CONTRIBUTING FACTOR VEHICLE 5    2258873
COLLISION_ID                           0
VEHICLE TYPE COD

In [10]:
# Comparing the Changes
null_summary = pd.DataFrame({
    "null_count": clean_df.isna().sum(),
    "null_percentage": clean_df.isna().mean() * 100
})

null_summary.sort_values("null_percentage", ascending=False)

,null_count,null_percentage
VEHICLE TYPE CODE 5,2259199,99.559842
CONTRIBUTING FACTOR VEHICLE 5,2258873,99.545476
VEHICLE TYPE CODE 4,2232943,98.402776
CONTRIBUTING FACTOR VEHICLE 4,2231554,98.341565
VEHICLE TYPE CODE 3,2111357,93.044646
CONTRIBUTING FACTOR VEHICLE 3,2104798,92.755599
OFF STREET NAME,1862411,82.073932
CROSS STREET NAME,870252,38.350828
ZIP CODE,691703,30.482415
BOROUGH,691375,30.467961


**2. Geographical Cleaning**

In [11]:
valid_location = (
    clean_df["LATITUDE"].notna()
    & clean_df["LONGITUDE"].notna()
    & clean_df["LATITUDE"].between(-90, 90)
    & clean_df["LONGITUDE"].between(-180, 180)
)

In [12]:
valid_location.value_counts()

True     2028275
False     240912
Name: count, dtype: int64

In [ ]:
clean_df = clean_df.loc[valid_location].copy() # Removed Records with Invalid Location

In [18]:
print(f"Shape after geographic filtering: {clean_df.shape}")
print("Missing latitude :", clean_df["LATITUDE"].isna().sum())
print("Missing longitude:", clean_df["LONGITUDE"].isna().sum())

Shape after geographic filtering: (2028275, 29)
Missing latitude : 0
Missing longitude: 0


**3. Cleaning Date/Time**

In [19]:
clean_df[['CRASH DATE','CRASH TIME']].head(10)

,CRASH DATE,CRASH TIME
2,11/01/2023,1:29
9,09/11/2021,9:35
10,12/14/2021,8:13
12,12/14/2021,17:05
13,12/14/2021,8:17
14,12/14/2021,21:10
15,12/14/2021,14:58
16,12/13/2021,0:34
17,12/14/2021,16:50
19,12/14/2021,0:59


In [20]:
print(clean_df["CRASH DATE"].dtype)
print(clean_df["CRASH TIME"].dtype)

str
str


In [21]:
# Convert to datetime
clean_df["CRASH DATE"] = pd.to_datetime(
    clean_df["CRASH DATE"],
    errors="coerce"
)

In [22]:
invalid_dates = clean_df["CRASH DATE"].isna().sum()

print(f"Invalid/missing crash dates: {invalid_dates:,}")

Invalid/missing crash dates: 0


In [23]:
# Validate the Range
print("Minimum crash date:", clean_df["CRASH DATE"].min())
print("Maximum crash date:", clean_df["CRASH DATE"].max())

Minimum crash date: 2012-07-01 00:00:00
Maximum crash date: 2026-06-11 00:00:00


In [24]:
crash_datetime = pd.to_datetime(
    clean_df["CRASH DATE"].dt.strftime("%Y-%m-%d")
    + " "
    + clean_df["CRASH TIME"].astype("string"),
    errors="coerce"
)

In [25]:
invalid_datetime = crash_datetime.isna().sum()

print(f"Invalid crash date/time combinations: {invalid_datetime:,}")

Invalid crash date/time combinations: 0


In [27]:
# Add this to the dataset
clean_df["CRASH_DATETIME"] = crash_datetime

In [29]:
# Create Crash Hour
clean_df["CRASH_HOUR"] = clean_df["CRASH_DATETIME"].dt.hour
print("Minimum hour:", clean_df["CRASH_HOUR"].min())
print("Maximum hour:", clean_df["CRASH_HOUR"].max())

Minimum hour: 0
Maximum hour: 23


In [31]:
# Checking for Impossible Datetime Possibility
invalid_temporal = (
    clean_df["CRASH DATE"].isna()
    | clean_df["CRASH_DATETIME"].isna()
    | ~clean_df["CRASH_HOUR"].between(0, 23)
)

print(f"Invalid temporal records: {invalid_temporal.sum():,}")

Invalid temporal records: 0


In [32]:
# Creating More Temporal Columns
clean_df["CRASH_YEAR"] = clean_df["CRASH_DATETIME"].dt.year

clean_df["CRASH_QUARTER"] = clean_df["CRASH_DATETIME"].dt.quarter

clean_df["CRASH_MONTH"] = clean_df["CRASH_DATETIME"].dt.month

clean_df["CRASH_MONTH_NAME"] = clean_df["CRASH_DATETIME"].dt.month_name()

clean_df["CRASH_DAY"] = clean_df["CRASH_DATETIME"].dt.day

clean_df["CRASH_DAY_OF_WEEK"] = clean_df["CRASH_DATETIME"].dt.dayofweek

clean_df["CRASH_DAY_NAME"] = clean_df["CRASH_DATETIME"].dt.day_name()

clean_df["IS_WEEKEND"] = (
    clean_df["CRASH_DAY_OF_WEEK"] >= 5
)

clean_df["CRASH_WEEK"] = clean_df["CRASH_DATETIME"].dt.isocalendar().week

clean_df["DATE_KEY"] = (
    clean_df["CRASH_DATETIME"].dt.strftime("%Y%m%d").astype(int)
)

In [34]:
# Classifying the Time of the Day

def classify_time_of_day(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

In [35]:
clean_df["TIME_OF_DAY"] = clean_df["CRASH_HOUR"].apply(classify_time_of_day)

In [36]:
clean_df["TIME_OF_DAY"].value_counts()

TIME_OF_DAY
Afternoon    770045
Morning      528791
Evening      513114
Night        216325
Name: count, dtype: int64

**4. Numeric Data Validation**

In [ ]:
numeric_columns = [
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED",
    "ZIP CODE"
]

In [44]:
clean_df[numeric_columns].dtypes

NUMBER OF PERSONS INJURED        float64
NUMBER OF PERSONS KILLED         float64
NUMBER OF PEDESTRIANS INJURED      int64
NUMBER OF PEDESTRIANS KILLED       int64
NUMBER OF CYCLIST INJURED          int64
NUMBER OF CYCLIST KILLED           int64
NUMBER OF MOTORIST INJURED         int64
NUMBER OF MOTORIST KILLED          int64
ZIP CODE                          object
dtype: object

In [45]:
numeric_nulls = pd.DataFrame({
    "null_count": clean_df[numeric_columns].isna().sum(),
    "null_percentage": clean_df[numeric_columns].isna().mean() * 100
})

numeric_nulls.sort_values("null_count", ascending=False)

,null_count,null_percentage
ZIP CODE,488640,24.091408
NUMBER OF PERSONS KILLED,8852,0.436430
NUMBER OF PERSONS INJURED,16,0.000789
NUMBER OF PEDESTRIANS KILLED,0,0.000000
NUMBER OF PEDESTRIANS INJURED,0,0.000000
NUMBER OF CYCLIST INJURED,0,0.000000
NUMBER OF CYCLIST KILLED,0,0.000000
NUMBER OF MOTORIST INJURED,0,0.000000
NUMBER OF MOTORIST KILLED,0,0.000000


In [47]:
clean_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
NUMBER OF PERSONS INJURED,2028259.0,0.336019,0.716787,0.0,0.0,0.0,0.0,43.0
NUMBER OF PERSONS KILLED,2019423.0,0.001568,0.041774,0.0,0.0,0.0,0.0,8.0
NUMBER OF PEDESTRIANS INJURED,2028275.0,0.063622,0.257732,0.0,0.0,0.0,0.0,27.0
NUMBER OF PEDESTRIANS KILLED,2028275.0,0.000797,0.028897,0.0,0.0,0.0,0.0,6.0
NUMBER OF CYCLIST INJURED,2028275.0,0.031039,0.175682,0.0,0.0,0.0,0.0,4.0
NUMBER OF CYCLIST KILLED,2028275.0,0.000129,0.011386,0.0,0.0,0.0,0.0,2.0
NUMBER OF MOTORIST INJURED,2028275.0,0.236154,0.676038,0.0,0.0,0.0,0.0,43.0
NUMBER OF MOTORIST KILLED,2028275.0,0.000611,0.026978,0.0,0.0,0.0,0.0,5.0


In [49]:
severity_columns = [
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED"
]

In [50]:
for column in severity_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].nlargest(10).to_list())


--- NUMBER OF PERSONS INJURED ---
[43.0, 34.0, 32.0, 27.0, 25.0, 24.0, 24.0, 24.0, 23.0, 22.0]

--- NUMBER OF PERSONS KILLED ---
[8.0, 5.0, 4.0, 4.0, 4.0, 4.0, 3.0, 3.0, 3.0, 3.0]

--- NUMBER OF PEDESTRIANS INJURED ---
[27, 19, 15, 13, 9, 9, 8, 8, 7, 7]

--- NUMBER OF PEDESTRIANS KILLED ---
[6, 4, 3, 2, 2, 2, 2, 2, 2, 2]

--- NUMBER OF CYCLIST INJURED ---
[4, 3, 3, 3, 3, 3, 3, 3, 3, 3]

--- NUMBER OF CYCLIST KILLED ---
[2, 1, 1, 1, 1, 1, 1, 1, 1, 1]

--- NUMBER OF MOTORIST INJURED ---
[43, 34, 30, 25, 24, 24, 24, 23, 22, 22]

--- NUMBER OF MOTORIST KILLED ---
[5, 4, 4, 3, 3, 3, 3, 3, 3, 3]


In [51]:
clean_df["CALCULATED_PERSONS_INJURED"] = (
    clean_df["NUMBER OF PEDESTRIANS INJURED"]
    + clean_df["NUMBER OF CYCLIST INJURED"]
    + clean_df["NUMBER OF MOTORIST INJURED"]
)

In [52]:
injury_mismatch = (
    clean_df["NUMBER OF PERSONS INJURED"]
    != clean_df["CALCULATED_PERSONS_INJURED"]
)

print(f"Injury mismatches: {injury_mismatch.sum():,}")

Injury mismatches: 10,328


In [53]:
clean_df["CALCULATED_PERSONS_KILLED"] = (
    clean_df["NUMBER OF PEDESTRIANS KILLED"]
    + clean_df["NUMBER OF CYCLIST KILLED"]
    + clean_df["NUMBER OF MOTORIST KILLED"]
)
fatality_mismatch = (
    clean_df["NUMBER OF PERSONS KILLED"]
    != clean_df["CALCULATED_PERSONS_KILLED"]
)

print(f"Fatality mismatches: {fatality_mismatch.sum():,}")

Fatality mismatches: 8,929


**Cleaning ZIP CODES:**

In [55]:
clean_df["ZIP CODE"] = clean_df["ZIP CODE"].astype("string")

In [57]:
zip_mask = (
    clean_df["ZIP CODE"].notna()
    & ~clean_df["ZIP CODE"].str.fullmatch(r"\d{5}")
)

print(f"Invalid ZIP code format: {zip_mask.sum():,}")

Invalid ZIP code format: 1,160,147


In [59]:
clean_df.drop(['CALCULATED_PERSONS_KILLED','CALCULATED_PERSONS_INJURED'] , axis=1,inplace=True)

In [60]:
zip_inspection = (
    clean_df["ZIP CODE"]
    .value_counts(dropna=False)
    .rename_axis("ZIP_CODE")
    .reset_index(name="COUNT")
)

zip_inspection.head(50)

,ZIP_CODE,COUNT
0,<NA>,488640
1,11207.0,23824
2,11236.0,17130
3,11203.0,15577
4,11208.0,15291
5,11212.0,15256
6,11234.0,15255
7,11385.0,15008
8,11101.0,14869
9,11226.0,14538


In [62]:
# Clean the ZIP CODES
clean_df["ZIP CODE"] = (
    clean_df["ZIP CODE"]
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

In [65]:
clean_df["ZIP CODE"].value_counts(dropna=False).head(30)

ZIP CODE
<NA>     488640
11207     30538
11236     21265
11101     20499
11203     20139
11234     19447
11385     19210
11208     19061
11212     18847
11201     18557
11226     18500
10016     17430
11434     17291
10019     17078
10036     16744
10001     16425
10002     16367
11233     16343
10022     16342
11206     15853
10013     15391
10467     15264
11220     15143
11368     15132
11211     14883
11230     14838
11377     14012
11373     13991
11235     13541
11213     13403
Name: count, dtype: int64[pyarrow]

In [67]:
zip_mask = (
    clean_df["ZIP CODE"].notna()
    & ~clean_df["ZIP CODE"].str.fullmatch(r"\d{5}")
)
print(f"Invalid ZIP code format: {zip_mask.sum():,}")

Invalid ZIP code format: 0


- Now There Are Zero Invalid ZIP Codes

In [68]:
# Checking the Total Injuries or Fatalities are Whole Numbers
severity_columns = [
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED"
]

decimal_counts = {}

for column in severity_columns:
    values = clean_df[column].dropna()
    decimal_counts[column] = (values % 1 != 0).sum()

decimal_counts

{'NUMBER OF PERSONS INJURED': np.int64(0),
 'NUMBER OF PERSONS KILLED': np.int64(0),
 'NUMBER OF PEDESTRIANS INJURED': np.int64(0),
 'NUMBER OF PEDESTRIANS KILLED': np.int64(0),
 'NUMBER OF CYCLIST INJURED': np.int64(0),
 'NUMBER OF CYCLIST KILLED': np.int64(0),
 'NUMBER OF MOTORIST INJURED': np.int64(0),
 'NUMBER OF MOTORIST KILLED': np.int64(0)}

- These are logically counts, not continuous measurements.

In [69]:
# Converting to Nullable Integer
for column in severity_columns:
    clean_df[column] = clean_df[column].astype("Int64")

In [72]:
numeric_validation = pd.DataFrame({
    "null_count": clean_df[severity_columns].isna().sum(),
    "min": clean_df[severity_columns].min(),
    "max": clean_df[severity_columns].max()
})

numeric_validation



,null_count,min,max
NUMBER OF PERSONS INJURED,16,0,43
NUMBER OF PERSONS KILLED,8852,0,8
NUMBER OF PEDESTRIANS INJURED,0,0,27
NUMBER OF PEDESTRIANS KILLED,0,0,6
NUMBER OF CYCLIST INJURED,0,0,4
NUMBER OF CYCLIST KILLED,0,0,2
NUMBER OF MOTORIST INJURED,0,0,43
NUMBER OF MOTORIST KILLED,0,0,5


**5. Cleaning Textual Columns**

**5.1 BOROUGH**

In [ ]:
clean_df["BOROUGH"].value_counts(dropna=False)

BOROUGH
BROOKLYN         496369
NaN              488338
QUEENS           413687
MANHATTAN        338388
BRONX            227119
STATEN ISLAND     64374
Name: count, dtype: int64

In [77]:
clean_df["BOROUGH"] = (
    clean_df["BOROUGH"]
    .str.strip()
)

In [79]:
clean_df["BOROUGH"] = (
    clean_df["BOROUGH"]
    .str.upper()
)

In [80]:
clean_df["BOROUGH"].value_counts(dropna=False)

BOROUGH
BROOKLYN         496369
NaN              488338
QUEENS           413687
MANHATTAN        338388
BRONX            227119
STATEN ISLAND     64374
Name: count, dtype: int64

In [82]:
clean_df['BOROUGH'] = clean_df['BOROUGH'].fillna('UNKNOWN')

**5.2 Street Names**

In [84]:
street_columns = [
    "ON STREET NAME",
    "CROSS STREET NAME",
    "OFF STREET NAME"
]

for column in street_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", clean_df[column].nunique(dropna=True))
    print("Missing:", clean_df[column].isna().sum())


--- ON STREET NAME ---
Unique values: 16931
Missing: 442505

--- CROSS STREET NAME ---
Unique values: 19175
Missing: 773256

--- OFF STREET NAME ---
Unique values: 253867
Missing: 1647848


In [85]:
for column in street_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].dropna().sample(20, random_state=42).tolist())


--- ON STREET NAME ---
['31 PLACE                        ', 'MYRTLE AVENUE', '59 STREET                       ', 'HYLAN BOULEVARD                 ', 'UTICA AVE', 'BRIGGS AVENUE', 'MOORE STREET                    ', 'BROADWAY                        ', 'ROCKAWAY BOULEVARD              ', 'EAST 233 STREET                 ', 'EAST 135 STREET', '65 AVENUE                       ', 'CHESTNUT STREET', 'RYDER AVENUE                    ', 'ALLEN STREET                    ', 'UTICA AVENUE                    ', 'EAST 96 STREET                  ', 'CANAL STREET                    ', 'HIGHLAND AVENUE', 'BELT PARKWAY                    ']

--- CROSS STREET NAME ---
['PARK AVENUE', 'WEST 47 STREET                  ', 'NORTH 6 STREET                  ', '12 AVENUE', 'WEST 178 STREET', 'DELANCEY STREET', '116 AVE', '65 PLACE', 'STANLEY AVENUE                  ', 'ATLANTIC AVENUE                 ', '65 ROAD', 'WESTCHESTER AVENUE', 'WHITE PLAINS ROAD', 'QUEENS BOULEVARD', '7 AVENUE                       

In [86]:
# Remove Trailing and Leading Whitespaces
for column in street_columns:
    clean_df[column] = clean_df[column].str.strip()

In [87]:
for column in street_columns:
    clean_df[column] = (
        clean_df[column]
        .str.replace(r"\s+", " ", regex=True)
    )

In [88]:
for column in street_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].dropna().sample(20, random_state=42).tolist())


--- ON STREET NAME ---
['31 PLACE', 'MYRTLE AVENUE', '59 STREET', 'HYLAN BOULEVARD', 'UTICA AVE', 'BRIGGS AVENUE', 'MOORE STREET', 'BROADWAY', 'ROCKAWAY BOULEVARD', 'EAST 233 STREET', 'EAST 135 STREET', '65 AVENUE', 'CHESTNUT STREET', 'RYDER AVENUE', 'ALLEN STREET', 'UTICA AVENUE', 'EAST 96 STREET', 'CANAL STREET', 'HIGHLAND AVENUE', 'BELT PARKWAY']

--- CROSS STREET NAME ---
['PARK AVENUE', 'WEST 47 STREET', 'NORTH 6 STREET', '12 AVENUE', 'WEST 178 STREET', 'DELANCEY STREET', '116 AVE', '65 PLACE', 'STANLEY AVENUE', 'ATLANTIC AVENUE', '65 ROAD', 'WESTCHESTER AVENUE', 'WHITE PLAINS ROAD', 'QUEENS BOULEVARD', '7 AVENUE', '135 ST', 'BUSHWICK AVENUE', 'OCEAN PARKWAY', 'EUCLID AVENUE', 'QUEENS BOULEVARD']

--- OFF STREET NAME ---
['89-04 JAMAICA AVENUE', '53-43 96 STREET', '24-04 24 STREET', '111-51 116 STREET', '1166 GLENMORE AVENUE', '1184 HALSEY STREET', '4015 7 AVENUE', '40 WEST 225 STREET', '135 WEST 95 STREET', '86-20 KINGSTON PLACE', '401 7 AVENUE', '92-33 168 ST', '1543 MYRTLE AVE

In [90]:
clean_df[
    [
        "LOCATION",
        "LATITUDE",
        "LONGITUDE",
        "ON STREET NAME",
        "CROSS STREET NAME",
        "OFF STREET NAME"
    ]
].sample(20, random_state=42)

,LOCATION,LATITUDE,LONGITUDE,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME
2215240,"(40.79157, -73.94468)",40.791570,-73.944680,E 106 ST,3 AVE,NaN
2144712,"(40.68324, -73.98319)",40.683240,-73.983190,NaN,NaN,277 WYCKOFF ST
1625165,"(40.6795011, -73.9972816)",40.679501,-73.997282,2 PLACE,COURT STREET,NaN
1219942,"(40.672638, -73.935295)",40.672638,-73.935295,NaN,NaN,1267 PARK PLACE
1782906,"(40.7460687, -73.8836476)",40.746069,-73.883648,82 STREET,BAXTER AVENUE,NaN
1888328,"(40.7845371, -73.9735961)",40.784537,-73.973596,COLUMBUS AVENUE,WEST 83 STREET,NaN
1039660,"(40.744576, -73.82581)",40.744576,-73.825810,58 ROAD,MAIN STREET,NaN
1302590,"(40.728992, -73.95067)",40.728992,-73.950670,CALYER STREET,MC GUINNESS BOULEVARD,NaN
869978,"(40.67661, -73.77677)",40.676610,-73.776770,NaN,NaN,163-29 130 AVENUE
190080,"(40.67413, -73.89541)",40.674130,-73.895410,NaN,NaN,224 NEW JERSEY AVENUE


In [91]:
clean_df.drop(columns=["LOCATION"], inplace=True)

In [92]:
vehicle_columns = [
    "VEHICLE TYPE CODE 1",
    "VEHICLE TYPE CODE 2",
    "VEHICLE TYPE CODE 3",
    "VEHICLE TYPE CODE 4",
    "VEHICLE TYPE CODE 5"
]

factor_columns = [
    "CONTRIBUTING FACTOR VEHICLE 1",
    "CONTRIBUTING FACTOR VEHICLE 2",
    "CONTRIBUTING FACTOR VEHICLE 3",
    "CONTRIBUTING FACTOR VEHICLE 4",
    "CONTRIBUTING FACTOR VEHICLE 5"
]

In [93]:
for column in vehicle_columns + factor_columns:
    clean_df[column] = (
        clean_df[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [94]:
for column in vehicle_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].value_counts(dropna=False).head(20))


--- VEHICLE TYPE CODE 1 ---
VEHICLE TYPE CODE 1
Sedan                                  614946
Station Wagon/Sport Utility Vehicle    478122
PASSENGER VEHICLE                      347162
SPORT UTILITY / STATION WAGON          150919
Taxi                                    53099
Pick-up Truck                           36261
4 dr sedan                              32284
TAXI                                    29376
Box Truck                               25180
Bus                                     23385
VAN                                     22006
OTHER                                   19563
UNKNOWN                                 17211
Bike                                    16853
NaN                                     15817
BUS                                     12297
SMALL COM VEH(4 TIRES)                  11460
LARGE COM VEH(6 OR MORE TIRES)          11424
Tractor Truck Diesel                    10262
PICK-UP TRUCK                            9644
Name: count, dtype: int64

--- 

In [95]:
for column in factor_columns:
    print(f"\n--- {column} ---")
    print(clean_df[column].value_counts(dropna=False).head(20))


--- CONTRIBUTING FACTOR VEHICLE 1 ---
CONTRIBUTING FACTOR VEHICLE 1
Unspecified                       671539
Driver Inattention/Distraction    419915
Failure to Yield Right-of-Way     124976
Following Too Closely             105233
Backing Unsafely                   75964
Other Vehicular                    63994
Passing or Lane Usage Improper     59631
Passing Too Closely                54711
Turning Improperly                 48777
Unsafe Lane Changing               38613
Fatigued/Drowsy                    37937
Traffic Control Disregarded        37778
Driver Inexperience                33467
Unsafe Speed                       32399
Alcohol Involvement                24103
Reaction to Uninvolved Vehicle     19518
Pavement Slippery                  18318
Lost Consciousness                 17849
View Obstructed/Limited            14458
Prescription Medication            13067
Name: count, dtype: int64

--- CONTRIBUTING FACTOR VEHICLE 2 ---
CONTRIBUTING FACTOR VEHICLE 2
Unspecified     

In [96]:
vehicle_profile = pd.concat(
    [
        clean_df[column]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
        .assign(source_column=column)
        .reset_index(names="vehicle_type")
        for column in vehicle_columns
    ],
    ignore_index=True
)

vehicle_profile = vehicle_profile.sort_values(
    "count",
    ascending=False
)

vehicle_profile.head(50)

,vehicle_type,count,source_column
4229,NaN,2019146,VEHICLE TYPE CODE 5
4110,NaN,1995664,VEHICLE TYPE CODE 4
3806,NaN,1887911,VEHICLE TYPE CODE 3
0,Sedan,614946,VEHICLE TYPE CODE 1
1,Station Wagon/Sport Utility Vehicle,478122,VEHICLE TYPE CODE 1
1793,NaN,426313,VEHICLE TYPE CODE 2
1794,Sedan,416943,VEHICLE TYPE CODE 2
2,PASSENGER VEHICLE,347162,VEHICLE TYPE CODE 1
1795,Station Wagon/Sport Utility Vehicle,333263,VEHICLE TYPE CODE 2
1796,PASSENGER VEHICLE,263677,VEHICLE TYPE CODE 2


In [97]:
for column in vehicle_columns:
    clean_df[column] = (
        clean_df[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

In [99]:
all_vehicle_values = (
    clean_df[vehicle_columns]
    .stack()
    .dropna()
)

vehicle_counts = (
    all_vehicle_values
    .value_counts()
    .rename_axis("vehicle_type")
    .reset_index(name="count")
)

print(f"Unique vehicle values: {vehicle_counts['vehicle_type'].nunique():,}")
vehicle_counts.head(100)

Unique vehicle values: 2,043


,vehicle_type,count
0,SEDAN,1094299
1,STATION WAGON/SPORT UTILITY VEHICLE,862705
2,PASSENGER VEHICLE,640442
3,SPORT UTILITY / STATION WAGON,282299
4,TAXI,148244
...,...,...
95,FDNY AMBUL,115
96,FDNY FIRE,113
97,FLAT,109
98,UTILI,108


**Contributing Factor Cleaning**

In [101]:
factor_columns = [
    "CONTRIBUTING FACTOR VEHICLE 1",
    "CONTRIBUTING FACTOR VEHICLE 2",
    "CONTRIBUTING FACTOR VEHICLE 3",
    "CONTRIBUTING FACTOR VEHICLE 4",
    "CONTRIBUTING FACTOR VEHICLE 5"
]

In [102]:
for column in factor_columns:
    clean_df[column] = (
        clean_df[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

In [103]:
all_factor_values = (
    clean_df[factor_columns]
    .stack()
    .dropna()
)

In [104]:
factor_counts = (
    all_factor_values
    .value_counts()
    .rename_axis("factor")
    .reset_index(name="count")
)

In [105]:
print(
    f"Unique contributing factors: "
    f"{factor_counts['factor'].nunique():,}"
)

Unique contributing factors: 64


In [106]:
factor_counts.head(50)

,factor,count
0,UNSPECIFIED,2274102
1,DRIVER INATTENTION/DISTRACTION,514465
2,FAILURE TO YIELD RIGHT-OF-WAY,141456
3,FOLLOWING TOO CLOSELY,125851
4,OTHER VEHICULAR,99030
5,BACKING UNSAFELY,83445
6,PASSING OR LANE USAGE IMPROPER,72456
7,PASSING TOO CLOSELY,63626
8,TURNING IMPROPERLY,56773
9,FATIGUED/DROWSY,47305


In [113]:
factor_counts.tail(50)

,factor,count
14,ALCOHOL INVOLVEMENT,25790
15,REACTION TO UNINVOLVED VEHICLE,22965
16,PAVEMENT SLIPPERY,22574
17,LOST CONSCIOUSNESS,22284
18,VIEW OBSTRUCTED/LIMITED,17728
19,PRESCRIPTION MEDICATION,15638
20,PEDESTRIAN/BICYCLIST/OTHER PEDESTRIAN ERROR/CO...,14183
21,OVERSIZED VEHICLE,14070
22,OUTSIDE CAR DISTRACTION,13392
23,AGGRESSIVE DRIVING/ROAD RAGE,12126


### Final Validation

In [119]:
duplicate_ids = clean_df["COLLISION_ID"].duplicated().sum()

print(f"Duplicate COLLISION_IDs: {duplicate_ids:,}")

Duplicate COLLISION_IDs: 0


In [120]:
coordinate_nulls = clean_df[
    ["LATITUDE", "LONGITUDE"]
].isna().sum()

coordinate_nulls

LATITUDE     0
LONGITUDE    0
dtype: int64

In [121]:
zip_invalid = (
    clean_df["ZIP CODE"].notna()
    & ~clean_df["ZIP CODE"].str.fullmatch(r"\d{5}")
)

print(f"Invalid ZIP codes: {zip_invalid.sum():,}")

Invalid ZIP codes: 0


In [122]:
print(
    "Missing CRASH_DATETIME:",
    clean_df["CRASH_DATETIME"].isna().sum()
)

Missing CRASH_DATETIME: 0


In [123]:
clean_df["BOROUGH"].value_counts(dropna=False)

BOROUGH
BROOKLYN         496369
UNKNOWN          488338
QUEENS           413687
MANHATTAN        338388
BRONX            227119
STATEN ISLAND     64374
Name: count, dtype: int64

In [124]:
print("Rows:", f"{len(clean_df):,}")
print("Columns:", len(clean_df.columns))

Rows: 2,028,275
Columns: 41


In [125]:
final_null_profile = pd.DataFrame({
    "null_count": clean_df.isna().sum(),
    "null_percentage": (
        clean_df.isna().mean() * 100
    )
})

final_null_profile = (
    final_null_profile
    .sort_values("null_count", ascending=False)
)

final_null_profile

,null_count,null_percentage
VEHICLE TYPE CODE 5,2019146,99.549913
CONTRIBUTING FACTOR VEHICLE 5,2018856,99.535615
VEHICLE TYPE CODE 4,1995664,98.392181
CONTRIBUTING FACTOR VEHICLE 4,1994406,98.330157
VEHICLE TYPE CODE 3,1887911,93.079637
CONTRIBUTING FACTOR VEHICLE 3,1881939,92.785199
OFF STREET NAME,1647848,81.243816
CROSS STREET NAME,773256,38.123824
ZIP CODE,488640,24.091408
ON STREET NAME,442505,21.816815


In [127]:
cleaned_path = "D:/Study/NYC_Vehicle_Collision_Project/data/nyc_crash_data_cleaned.csv"

clean_df.to_csv(
    cleaned_path,
    index=False
)

print(f"Saved cleaned dataset to: {cleaned_path}")

Saved cleaned dataset to: D:/Study/NYC_Vehicle_Collision_Project/data/nyc_crash_data_cleaned.csv


In [128]:
validation_df = pd.read_csv(cleaned_path)

print("Saved file shape:", validation_df.shape)

Saved file shape: (2028275, 41)


In [129]:
validation_df.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,...,CRASH_QUARTER,CRASH_MONTH,CRASH_MONTH_NAME,CRASH_DAY,CRASH_DAY_OF_WEEK,CRASH_DAY_NAME,IS_WEEKEND,CRASH_WEEK,DATE_KEY,TIME_OF_DAY
0,2023-11-01,1:29,BROOKLYN,11230.0,40.621790,-73.970024,OCEAN PARKWAY,AVENUE K,NaN,1.0,...,4,11,November,1,2,Wednesday,False,44,20231101,Night
1,2021-09-11,9:35,BROOKLYN,11208.0,40.667202,-73.866500,NaN,NaN,1211 LORING AVENUE,0.0,...,3,9,September,11,5,Saturday,True,36,20210911,Morning
2,2021-12-14,8:13,BROOKLYN,11233.0,40.683304,-73.917274,SARATOGA AVENUE,DECATUR STREET,NaN,0.0,...,4,12,December,14,1,Tuesday,False,50,20211214,Morning
3,2021-12-14,17:05,UNKNOWN,NaN,40.709183,-73.956825,BROOKLYN QUEENS EXPRESSWAY,NaN,NaN,0.0,...,4,12,December,14,1,Tuesday,False,50,20211214,Afternoon
4,2021-12-14,8:17,BRONX,10475.0,40.868160,-73.831480,NaN,NaN,344 BAYCHESTER AVENUE,2.0,...,4,12,December,14,1,Tuesday,False,50,20211214,Morning


In [130]:
validation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2028275 entries, 0 to 2028274
Data columns (total 41 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   CRASH DATE                     str    
 1   CRASH TIME                     str    
 2   BOROUGH                        str    
 3   ZIP CODE                       float64
 4   LATITUDE                       float64
 5   LONGITUDE                      float64
 6   ON STREET NAME                 str    
 7   CROSS STREET NAME              str    
 8   OFF STREET NAME                str    
 9   NUMBER OF PERSONS INJURED      float64
 10  NUMBER OF PERSONS KILLED       float64
 11  NUMBER OF PEDESTRIANS INJURED  int64  
 12  NUMBER OF PEDESTRIANS KILLED   int64  
 13  NUMBER OF CYCLIST INJURED      int64  
 14  NUMBER OF CYCLIST KILLED       int64  
 15  NUMBER OF MOTORIST INJURED     int64  
 16  NUMBER OF MOTORIST KILLED      int64  
 17  CONTRIBUTING FACTOR VEHICLE 1  str    
 18  CONTRIBUTING 

# Done The Cleaning of RAW DATASET